# Runbook della pipeline (tesi UML + RAG)

Tutti i comandi della pipeline, in ordine, con un registro di ogni esecuzione (`data/results/run_log.jsonl`,
`experiments/run_log.py`, voce 102 di `docs/decisions.md`). Storico delle esecuzioni precedenti al registro:
`docs/runbook_history.md`.

**Sicurezze**
- `ESEGUI_GENERAZIONI = False`: le celle che chiamano l'LLM (generazioni, calibrazioni, smoke test) NON partono e
  stampano il comando.
- Le celle che producono una run (generazione o retrieval) rifiutano di partire se la cartella di output esiste già,
  salvo `--resume` esplicito nel comando.
- `SBLOCCA_TESTSET = False`: la sezione del Passo 3b non fa nulla. Nessun'altra cella può lanciare un comando che legge il
  test set per produrre risultati (`retrieval/run_testset.py`, `retrieval/hubness_report.py`, config con
  `split: testset`): `run()` li rifiuta fuori dalla sezione 3b.
- Eccezione dichiarata: le suite di test e i controlli di sanità (sezione 2) leggono i ground truth del test set solo per
  verificare il codice (separazione degli split, schema, conversione), senza generare né valutare: è la pratica in uso
  dal Passo 1.
- Le celle di test e di analisi non chiamano l'LLM e girano liberamente. Le analisi RISCRIVONO i `summary.md` versionati:
  di norma cambia solo la riga "Analisi eseguita sul commit ..." (vedi `docs/runbook_history.md`).

Il notebook si salva SENZA output (`nbstripout`, configurato come filtro git: vedi la cella finale).

In [ ]:
# Interruttori (lasciare False salvo decisione esplicita)
ESEGUI_GENERAZIONI = False   # chiamate all'LLM (LM Studio): generazioni, calibrazioni, smoke test
SBLOCCA_TESTSET = False      # Passo 3b: qualunque comando che produca risultati sul test set

import json, os, shlex, subprocess, sys, time, urllib.request
from pathlib import Path
import yaml

ROOT = Path.cwd() if (Path.cwd() / "CLAUDE.md").exists() else Path.cwd().parent
assert (ROOT / "CLAUDE.md").exists(), f"radice del repository non trovata da {Path.cwd()}"
sys.path.insert(0, str(ROOT / "experiments"))
import run_log  # noqa: E402

TESTSET_SCRIPTS = {"retrieval/run_testset.py", "retrieval/hubness_report.py"}


def reads_testset(args):
    """True se il comando produce risultati sul test set: script del retrieval sul test set o config con split testset."""
    if any(a.replace("\\", "/") in TESTSET_SCRIPTS for a in args):
        return True
    for a in args:
        if a.endswith((".yaml", ".yml")) and (ROOT / a).is_file():
            if (yaml.safe_load((ROOT / a).read_text(encoding="utf-8")) or {}).get("split", "testset") == "testset":
                return True
    return False


def run(cmd, *, genera=False, test_set=False, output=None):
    """Lancia `python <cmd>` dalla radice del repository, mostra l'output in streaming e registra l'esecuzione
    (eventi notebook_start / notebook_end in data/results/run_log.jsonl). Ritorna il codice di uscita o None se non
    eseguito. Un codice diverso da 0 ferma il notebook (eccezione)."""
    args = shlex.split(cmd, posix=True)
    resume = "--resume" in args
    if reads_testset(args) and not test_set:
        print(f"RIFIUTATO (legge il test set, solo dalla sezione 3b): python {cmd}")
        return None
    if test_set and not SBLOCCA_TESTSET:
        print(f"BLOCCATO (SBLOCCA_TESTSET = False): python {cmd}")
        return None
    if genera and not ESEGUI_GENERAZIONI:
        print(f"NON ESEGUITO (ESEGUI_GENERAZIONI = False): python {cmd}")
        return None
    if output and (ROOT / output).exists() and not resume:
        print(f"RIFIUTATO: {output} esiste gia' (aggiungere --resume per riprendere una run interrotta): python {cmd}")
        return None
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    print(f"$ python {cmd}", flush=True)
    with run_log.logged(str(ROOT / "notebooks" / "runbook.ipynb"), argv=args, event_prefix="notebook_"):
        if output:
            run_log.set_output(ROOT / output)
        proc = subprocess.Popen([sys.executable, *args], cwd=ROOT, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in proc.stdout:
            print(line, end="", flush=True)
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"comando fallito (codice {rc}): python {cmd}")
    return rc


print("radice:", ROOT, "| ESEGUI_GENERAZIONI =", ESEGUI_GENERAZIONI, "| SBLOCCA_TESTSET =", SBLOCCA_TESTSET)

## 1. Setup e verifica dell'ambiente
Versioni di Python e dei pacchetti di `requirements.txt`, Java e `plantuml.jar` (rendering, voce 100), stato git, LM
Studio raggiungibile e modello caricato (solo lettura di `GET /api/v1/models`, nessuna generazione). Atteso: Python 3.12,
pacchetti installati, PlantUML 1.2023.0; LM Studio può essere spento (solo un avviso).

In [ ]:
import importlib.metadata as md_
print("Python", sys.version.split()[0], "|", sys.executable)
for pkg in ("rank-bm25", "snowballstemmer", "tiktoken", "pyyaml", "jsonschema", "numpy", "pandas", "scikit-learn",
            "openpyxl", "pypdf", "pillow", "nbformat", "nbclient", "nbstripout"):
    try:
        print(f"  {pkg:16s} {md_.version(pkg)}")
    except md_.PackageNotFoundError:
        print(f"  {pkg:16s} NON INSTALLATO")
jar = ROOT / ".tools" / "plantuml-old.jar"
try:
    v = subprocess.run(["java", "-jar", str(jar), "-version"], capture_output=True, text=True, timeout=60).stdout
    print("PlantUML:", v.splitlines()[0] if v else "non disponibile")
except (OSError, subprocess.TimeoutExpired) as e:
    print("PlantUML: non disponibile", e)
head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
dirty = bool(subprocess.run(["git", "status", "--porcelain"], cwd=ROOT, capture_output=True, text=True).stdout.strip())
print("git:", head, "(modifiche non committate)" if dirty else "(pulito)")
try:
    with urllib.request.urlopen("http://localhost:1234/api/v1/models", timeout=3) as r:
        models = json.load(r).get("models", [])
    loaded = [(m.get("key"), i.get("config", {}).get("context_length")) for m in models for i in m.get("loaded_instances") or []]
    print("LM Studio raggiungibile; modelli caricati (contesto):", loaded or "nessuno")
except OSError as e:
    print("LM Studio non raggiungibile (normale se non serve generare):", e)

## 2. Test e controlli di sanità
Nessuna chiamata a un LLM. Atteso: ogni comando termina con codice 0 ("Tutti i test ... sono passati", "SANITY CHECK: OK").
Durata: qualche minuto (`generation/test_generation.py` è il più lungo).

In [ ]:
for cmd in ("corpus/test_apollon_convert.py", "retrieval/test_retrieval.py", "generation/test_generation.py",
            "generation/sanity_check.py", "generation/plantuml_sanity_check.py", "generation/compact_sanity_check.py"):
    run(cmd)

## 3. Retrieval (Passo 2, voci 57-61)
Leave-one-out sul corpus con la configurazione congelata (`retrieval/config_bm25.yaml`). La run storica esiste già:
la cella la rifiuta (la cartella esiste). Per una run nuova serve un run-id nuovo e una voce in decisions.md.
Il test set del retrieval (`run_testset.py`, `hubness_report.py`) è nella sezione 3b.

In [ ]:
run("retrieval/analyze_retrieval.py --run-id loo_2026-10-04_stop1", output="data/results/retrieval/loo_2026-10-04_stop1")

## 4. Selezioni degli esercizi (solo corpus)
Selezioni deterministiche: pilota (voce 75) e insieme di sviluppo (voce 92). Atteso: gli stessi `query_ids` dei config.

In [ ]:
run("experiments/select_pilot.py")
run("experiments/select_dev.py")

## 5. Esperimenti di sviluppo (solo corpus, leave-one-out)
Ogni cella chiama l'LLM: con `ESEGUI_GENERAZIONI = False` stampa i comandi. Con `True`: LM Studio aperto con il modello
della configurazione caricato (contesto e Flash Attention verificati dal runner all'avvio). Le run storiche esistono già:
la cella le rifiuta (riprendere una run interrotta: aggiungere `--resume`).

### 5.1 Pilota sulla temperatura (voci 75-77)
Gemma 4 12B QAT, bm25 k = 2, temperature 0 e 0.3, 36 generazioni, versione di configurazione 1.
Atteso: una cartella per configurazione in `data/results/generation/` (manifest, raw, validation.csv, config.json) e due righe nel registro per comando.

In [ ]:
run("experiments/run_experiment.py experiments/configs/pilot_temperature.yaml", genera=True, output="data/results/generation/pilot_temperature_gemma4-12b-qat")

### 5.2 Secondo pilota: PlantUML contro JSON vincolato (voci 78-89)
Calibrazione dei token, poi una configurazione per volta (un modello caricato alla volta).
Atteso: una cartella per configurazione in `data/results/generation/` (manifest, raw, validation.csv, config.json) e due righe nel registro per comando.

In [ ]:
run("experiments/calibrate_tokens.py experiments/configs/pilot2_formats.yaml --configuration J-G", genera=True)
run("experiments/run_experiment.py experiments/configs/pilot2_formats.yaml --configuration P-G", genera=True, output="data/results/generation/pilot2_formats__P-G")
run("experiments/run_experiment.py experiments/configs/pilot2_formats.yaml --configuration J-G", genera=True, output="data/results/generation/pilot2_formats__J-G")
run("experiments/run_experiment.py experiments/configs/pilot2_formats.yaml --configuration P-Q", genera=True, output="data/results/generation/pilot2_formats__P-Q")
run("experiments/run_experiment.py experiments/configs/pilot2_formats.yaml --configuration J-Q", genera=True, output="data/results/generation/pilot2_formats__J-Q")
run("experiments/run_experiment.py experiments/configs/pilot2_formats.yaml --configuration J0-Q", genera=True, output="data/results/generation/pilot2_formats__J0-Q")

### 5.3 Insieme di sviluppo: PlantUML contro JSON compatto (voce 92)
20 esercizi del corpus, 160 generazioni, versione di configurazione 1.
Atteso: una cartella per configurazione in `data/results/generation/` (manifest, raw, validation.csv, config.json) e due righe nel registro per comando.

In [ ]:
run("experiments/calibrate_tokens.py experiments/configs/dev_formats.yaml --configuration C-G --formats plantuml compact", genera=True)
run("experiments/run_experiment.py experiments/configs/dev_formats.yaml --configuration P-G", genera=True, output="data/results/generation/dev_formats__P-G")
run("experiments/run_experiment.py experiments/configs/dev_formats.yaml --configuration C-G", genera=True, output="data/results/generation/dev_formats__C-G")
run("experiments/run_experiment.py experiments/configs/dev_formats.yaml --configuration P-Q", genera=True, output="data/results/generation/dev_formats__P-Q")
run("experiments/run_experiment.py experiments/configs/dev_formats.yaml --configuration C-Q", genera=True, output="data/results/generation/dev_formats__C-Q")

### 5.4 Leva k (voci 93-95)
k = 2, 3, 5, 8; versione di configurazione 2 (contesto 32768, max_tokens 4096); 640 generazioni.
Atteso: una cartella per configurazione in `data/results/generation/` (manifest, raw, validation.csv, config.json) e due righe nel registro per comando.

In [ ]:
run("experiments/calibrate_tokens.py experiments/configs/dev_k.yaml --configuration C-G --formats plantuml compact", genera=True)
run("experiments/run_experiment.py experiments/configs/dev_k.yaml --configuration P-G", genera=True, output="data/results/generation/dev_k__P-G")
run("experiments/run_experiment.py experiments/configs/dev_k.yaml --configuration C-G", genera=True, output="data/results/generation/dev_k__C-G")
run("experiments/run_experiment.py experiments/configs/dev_k.yaml --configuration P-Q", genera=True, output="data/results/generation/dev_k__P-Q")
run("experiments/run_experiment.py experiments/configs/dev_k.yaml --configuration C-Q", genera=True, output="data/results/generation/dev_k__C-Q")

### 5.5 Istruzioni mirate (voci 96-101)
Gemma, k = 3, blocco congelato di regole aggiuntive; 80 generazioni; esito: non adottate (voce 101).
Atteso: una cartella per configurazione in `data/results/generation/` (manifest, raw, validation.csv, config.json) e due righe nel registro per comando.

In [ ]:
run("experiments/run_experiment.py experiments/configs/dev_instructions.yaml --configuration P-G", genera=True, output="data/results/generation/dev_instructions__P-G")
run("experiments/run_experiment.py experiments/configs/dev_instructions.yaml --configuration C-G", genera=True, output="data/results/generation/dev_instructions__C-G")

## 6. Analisi (sola lettura delle run, nessuna chiamata a un LLM)
Ogni analisi ricalcola tutto dalle risposte grezze e riscrive il proprio `summary.md`. Atteso: gli stessi numeri dei
report versionati; cambia la riga "Analisi eseguita sul commit ...". `dev_k_analysis` differisce anche nel verso di
eHome2020 (GT corretto il 2026-10-09, voce 99).

In [ ]:
run("experiments/analyze_pilot.py")                         # voce 77
run("experiments/analyze_pilot2.py")                        # voce 88 (originale)
run("experiments/analyze_pilot2.py --postprocess v2")       # voce 89 (descrittiva)
run("experiments/analyze_dev.py")                           # voce 92
run("experiments/analyze_k.py")                             # voci 94-95
run("experiments/analyze_instructions.py")                  # voce 101

## 7. Report grafico di revisione (voce 100)
Pagina HTML autonoma con i diagrammi resi (plantuml.jar locale, layout smetana). Atteso:
`data/results/generation/dev_k_gemma_k3_review/index.html` (non versionato), circa 100 diagrammi in pochi secondi.

In [ ]:
run("experiments/review_report.py --runs dev_k__P-G dev_k__C-G --filter k=3 --name dev_k_gemma_k3")

## 8. Passo 3b — test set De Bari (BLOCCATO)
Il Passo 3b è bloccato in attesa dei relatori (domande 8, 9, 11, 14, 15, 16 di STATUS.md). Con `SBLOCCA_TESTSET = False`
questa cella non fa nulla. Il test set si guarda una volta sola: ogni run nuova richiede un run-id nuovo e una voce in
decisions.md; i template del 3b (`gemma4_12b_qat_template.yaml`, ...) hanno ancora dei TODO e vanno copiati in un config
apposito prima dell'uso.

In [ ]:
if not SBLOCCA_TESTSET:
    print("Passo 3b BLOCCATO (SBLOCCA_TESTSET = False): nessun comando eseguito.")
else:
    # retrieval sul test set (storico: run gia' presente, quindi rifiutata) e hubness (solo stampa)
    run("retrieval/run_testset.py --run-id testset_2026-10-04_stop2", test_set=True,
        output="data/results/retrieval/testset_2026-10-04_stop2")
    run("retrieval/hubness_report.py", test_set=True)
    # generazione sul test set: config da creare dal template (nessuno ancora pronto)
    print("Generazioni del 3b: creare experiments/configs/<run_id>.yaml dal template (TODO risolti) e lanciare\n"
          "run('experiments/run_experiment.py experiments/configs/<run_id>.yaml', genera=True, test_set=True, "
          "output='data/results/generation/<run_id>')")

## Manutenzione del notebook
Il notebook è versionato SENZA output. Installazione del filtro (una volta per clone):
`python -m pip install nbstripout` e `python -m nbstripout --install --attributes .gitattributes`
(la riga `*.ipynb filter=nbstripout` è già in `.gitattributes`; la configurazione del filtro sta in `.git/config`, che
non è versionato, quindi va ripetuta su ogni clone). Pulizia manuale: `python -m nbstripout notebooks/runbook.ipynb`.